# 00 · 环境自检与双后端接线

检测制造内核 :7862 与多模态层 :8866 在线状态；展示 TimoAdapter 的在线/离线 byte-identical 双路径。

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from bootstrap import build_controller, status
from services.config import load_settings, load_policy
print('trunk root:', ROOT)

In [ ]:
st = status()
print(json.dumps(st, ensure_ascii=False, indent=2))

In [ ]:
ctrl = build_controller()
print('制造内核:', ctrl.timo.source_label())
print('多模态层:', ctrl.funasr.source_label())

## 在线 vs 离线一致性证明

同一条 RFQ，分别走 live :7862 与离线 vendored kernel，结果应 byte-identical。

In [ ]:
rfq = {'material':'6061','surface':'阳极氧化','quantity':50,'weight_kg':0.5,'max_dim_mm':100,'tolerance':'IT7'}
live = ctrl.timo.quote(rfq)
print('LIVE  unit=%s final=%s src=%s' % (live.get('unit_price'), live.get('final_price'), live.get('_source')))
# 强制离线
s = load_settings(); s['timo']['base_url'] = 'http://127.0.0.1:59999'
from adapters.timo_adapter import TimoAdapter
off_ad = TimoAdapter(s)
off = off_ad.quote(rfq)
print('OFF   unit=%s final=%s src=%s' % (off.get('unit_price'), off.get('final_price'), off.get('_source')))

In [ ]:
cf_live = ctrl.timo.conflict_check('304','阳极氧化')
cf_off  = off_ad.conflict_check('304','阳极氧化')
print('304+阳极氧化 live valid=', cf_live['valid'], '| offline valid=', cf_off['valid'])
assert cf_live['valid'] == cf_off['valid'] == False, 'DFM 判定应一致且为硬冲突'
print('一致性断言通过 ✅')